# 1. Thiết lập lớp huấn luyện và đánh giá

In [1]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, confusion_matrix, classification_report
)
import numpy as np
import pandas as pd

class ClassificationMetrics:
    def compute(self, y_true, y_pred):
        metrics = {}

        # Overall metrics
        metrics["accuracy"] = accuracy_score(y_true, y_pred)
        metrics["balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)
        metrics["precision_macro"] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics["precision_weighted"] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics["recall_macro"] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics["recall_weighted"] = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics["f1_macro"] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics["f1_weighted"] = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        # Overall G-Mean (macro)
        recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
        metrics["gmean"] = np.prod(recalls) ** (1 / len(recalls))

        metrics["mcc"] = matthews_corrcoef(y_true, y_pred)
        metrics["kappa"] = cohen_kappa_score(y_true, y_pred)

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        metrics["confusion_matrix"] = cm

        # Per-class metrics
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

        # ---- Compute per-class G-Mean ----
        per_class_gmean = {}
        n_classes = cm.shape[0]
        for i in range(n_classes):
            TP = cm[i, i]
            FN = cm[i, :].sum() - TP
            FP = cm[:, i].sum() - TP
            TN = cm.sum() - (TP + FP + FN)

            TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
            TNR = TN / (TN + FP) if (TN + FP) > 0 else 0
            gmean_i = np.sqrt(TPR * TNR)

            per_class_gmean[str(i)] = gmean_i

        # Build per_class output
        metrics["per_class"] = {
            label: {
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1-score": report[label]["f1-score"],
                "support": report[label]["support"],
                "gmean": per_class_gmean[label]
            }
            for label in report if label.isdigit()
        }

        metrics["per_class_gmean"] = per_class_gmean

        return metrics


In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import ParameterGrid


class RFTrainer:
    def __init__(self, train_path):
        """
        train_path: đường dẫn file train CSV.
        """
        self.train_path = train_path
        self.train_df = None
        self.X_train = None
        self.y_train = None

        self.scaler = StandardScaler()
        self.model = None

        self.best_params = None
        self.best_score = None

    # -----------------------------------------------------
    def load_and_prepare_train(self):
        """Đọc dữ liệu train và xử lý cột"""
        self.train_df = pd.read_csv(self.train_path)

        X = self.train_df.drop(["label_3", "course_id", "user_id"], axis=1)
        y = self.train_df["label_3"]

        self.X_train = self.scaler.fit_transform(X)
        self.y_train = y

    # -----------------------------------------------------
    def _prepare_val_or_test(self, df):
        """
        Chuẩn hóa file val/test để phù hợp train
        - Bổ sung cột thiếu
        - Sắp xếp thứ tự
        - Chuẩn hóa
        """
        X_val = df.drop(["label_3", "course_id", "user_id"], axis=1)

        # Cột chuẩn từ train
        train_cols = self.train_df.drop(["label_3", "course_id", "user_id"], axis=1).columns

        # Xử lý thiếu cột
        missing_cols = train_cols.difference(X_val.columns)
        if len(missing_cols) > 0:
            for col in missing_cols:
                X_val[col] = self.train_df[col].median()

        # Đảm bảo đúng thứ tự
        X_val = X_val[train_cols]

        # Chuẩn hóa
        X_val = self.scaler.transform(X_val)

        return X_val

    # -----------------------------------------------------
    def train_with_gridsearch(self, val_path, param_grid):
        """
        Chạy GridSearch sử dụng F1-macro trên file validation.

        Args:
            val_path: path tới file validation
            param_grid: dict tham số RF

        Returns:
            best_model, best_params, best_f1
        """
        print("\n Running Grid Search using F1-macro...\n")

        val_df = pd.read_csv(val_path)
        y_val = val_df["label_3"]
        X_val = self._prepare_val_or_test(val_df)

        best_f1 = -1
        best_params = None
        best_model = None

        for params in ParameterGrid(param_grid):
            print(f"→ Testing params: {params}")

            model = RandomForestClassifier(
                **params,
                random_state=42,
                n_jobs=-1
            )

            model.fit(self.X_train, self.y_train)
            y_pred = model.predict(X_val)

            f1 = f1_score(y_val, y_pred, average="macro")
            print(f"   F1-macro = {f1:.4f}")

            if f1 > best_f1:
                best_f1 = f1
                best_params = params
                best_model = model

        # Lưu kết quả tốt nhất
        self.model = best_model
        self.best_params = best_params
        self.best_score = best_f1

        print("\n Best GridSearch Result:")
        print("Best Params:", best_params)
        print(f"Best F1-macro: {best_f1:.4f}")

        return best_model, best_params, best_f1

    # -----------------------------------------------------
    def evaluate_test(self, test_path, metrics):
        """
        Đánh giá model trên test set.
        """
        print(f"\n=== Evaluate Test Set: {test_path} ===\n")

        test_df = pd.read_csv(test_path)
        X_test = self._prepare_val_or_test(test_df)
        y_test = test_df["label_3"]

        y_pred = self.model.predict(X_test)

        results = metrics.compute(y_test, y_pred)

        print("Accuracy:", results["accuracy"])
        print("Macro F1:", results["f1_macro"])
        print("\nConfusion Matrix:")
        print(results["confusion_matrix"])
        print("\nPer-class Metrics:")
        print(results["per_class"])

        return results


# 2 Thực nghiệm

## 2.1 Thực nghiệm Random forest trên tập Raw

In [3]:
import json
import os
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/train.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [ 40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/raw"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search using F1-macro...

→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
   F1-macro = 0.8666
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
   F1-macro = 0.8688

 Best GridSearch Result:
Best Params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best F1-macro: 0.8688

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_1.csv ===

Accuracy: 0.9965498401827466
Macro F1: 0.3630745130943264

Confusion Matrix:
[[     1      2    220]
 [     0     26    579]
 [     0      1 231624]]

Per-class Metrics:
{'0': {'precision': 1.0, 'recall': 0.004484304932735426, 'f1-score': 0.008928571428571428, 'support': 223, 'gmean': 0.0669649530182425}, '1': {'precision': 0.896551724137931, 'recall': 0.04297520661157025, 'f1-score': 0.08201892744479496, 'support': 605, 'gmean': 0.20730328153062255}, '

## 2.2 Thực nghiệm Random forest trên tập data_median_sasmote

In [4]:
import json
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/train.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_median_sasmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search using F1-macro...

→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
   F1-macro = 0.6791
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
   F1-macro = 0.6799

 Best GridSearch Result:
Best Params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best F1-macro: 0.6799

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_1.csv ===

Accuracy: 0.007743500836728285
Macro F1: 0.00516102511713689

Confusion Matrix:
[[     0    223      0]
 [     0    593     12]
 [     0 230418   1207]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 223, 'gmean': 0.0}, '1': {'precision': 0.0025645017601217813, 'recall': 0.9801652892561984, 'f1-score': 0.005115619028722518, 'support': 605, 'gmean': 0.07143344865862958}, '2': {'precision': 0.9901558654634947, '

## 2.3 Thực nghiệm Random forest trên tập data_median_radiussmote

In [5]:
import json
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/train_median_radiussmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200,500],
    "max_depth": [40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_median_radiussmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search using F1-macro...

→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
   F1-macro = 0.2135
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
   F1-macro = 0.1965
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
   F1-macro = 0.2032

 Best GridSearch Result:
Best Params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Best F1-macro: 0.2135

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/test_1.csv ===

Accuracy: 0.0026026766701225624
Macro F1: 0.0017306135525634533

Confusion Matrix:
[[     0    223      0]
 [     0    605      0]
 [     0 231625      0]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 223, 'gmean': 0.0}, '1': {'precision': 0.0026026766701225624, 'recall': 1.0, 'f1-

## 2.4 Thực nghiệm trên tập data_median_cdsmote

In [7]:
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/train_median_cdsmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200,500],
    "max_depth": [40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_median_cdsmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search using F1-macro...

→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
   F1-macro = 0.1307
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
   F1-macro = 0.1293
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
   F1-macro = 0.1329

 Best GridSearch Result:
Best Params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
Best F1-macro: 0.1329

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_1.csv ===

Accuracy: 0.0026026766701225624
Macro F1: 0.0017306135525634533

Confusion Matrix:
[[     0    223      0]
 [     0    605      0]
 [     0 231625      0]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 223, 'gmean': 0.0}, '1': {'precision': 0.0026026766701225624, 'recall': 1.0, 'f1-scor

## 2.5 Thực nghiệm trên tập data_extra_trees_cdsmote

In [ ]:
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/train_extratrees_cdsmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200,500],
    "max_depth": [40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_extra_trees_cdsmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)

## 2.6 Thực nghiệm trên tập data_extra_trees_sasmote

In [10]:
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/train_extratrees_sasmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200,500],
    "max_depth": [40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_extra_trees_sasmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )
    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search using F1-macro...

→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
   F1-macro = 0.7047
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
   F1-macro = 0.7097
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
   F1-macro = 0.7104

 Best GridSearch Result:
Best Params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
Best F1-macro: 0.7104

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_1.csv ===

Accuracy: 0.11924561094070629
Macro F1: 0.07176710452656636

Confusion Matrix:
[[     0    223      0]
 [     0    560     45]
 [     0 204466  27159]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 223, 'gmean': 0.0}, '1': {'precision': 0.002728393317385225, 'recall': 0.92561983471

## 2.7 Thực nghiệm trên tập data_extra_trees_radiussmote

In [11]:
trainer = RFTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/train_extratrees_radiussmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [100, 200,500],
    "max_depth": [40],
    "min_samples_split": [5],
    "min_samples_leaf": [2]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_extra_trees_radiussmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )
    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search using F1-macro...

→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
   F1-macro = 0.2281
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
   F1-macro = 0.2300
→ Testing params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
   F1-macro = 0.2316

 Best GridSearch Result:
Best Params: {'max_depth': 40, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 500}
Best F1-macro: 0.2316

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_1.csv ===

Accuracy: 0.0026026766701225624
Macro F1: 0.0017306135525634533

Confusion Matrix:
[[     0    223      0]
 [     0    605      0]
 [     0 231625      0]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 223, 'gmean': 0.0}, '1': {'precision': 0.0026026766701225624, 'recall': 1.0,